# теория

#### 1. Cross‑Validation

##### Что такое Leave‑One‑Out?
- метод cv при котором каждую итерацию берем как train все строки кроме одной, она становится нашим test.
	- Плюсы: максимум данных в train, низкий bias
	- Минусы: очень медленно (N обучений), высокая variance


##### Когда нужна стратификация?
- Когда классы несбалансированы
- Нужно сохранить распределение таргета

##### Cильные/слабые стороны
Stratified K‑Fold  
- плюсы: Соблюден балланс классов, модель сможет увидеть зависимость, если она была
- минусы: Работает только для классификации, не решает проблему неразделимости некоторых классов(групп)

Stratified Group K‑Fold - комбинация stratification и group split
- плюсы: решает проблему выше, про группы которые нельзя делить. 
- минусы: если группы большие, по кол-ву элементов получися менее сбалансированно, только классификация 

##### Hyperparameter Optimization
Что такое параметры модели?
- параметры, которые модель обучает сама, пример: веса линейной регрессии
Что такое гиперпараметры?
- параметры, задаваемые до обучения. Примеры: alpha, n_estimators, l1_ratio

Как работает Grid Search?
- полный перебор всех комбинаций гиперпараметров, для каждой комбинации обучается модель и считается метрика на validation, выбирается комбинация с лучшей метрикой
- Минус: экспоненциальный рост числа комбинаций

Как работает Random Search?
- случайный выбор комбинаций параметров из заданных диапазонов, число итераций ограничено
- Плюс: быстрее grid search

Как работает Bayesian Optimization?
- строит вероятностную модель функции качества
- по acquisition функции выбирает следующую точку (на основании модели верхней)
- смотрит в ней метрики

3. Feature Selection
Классификация методов
- Filter, Wrapper, Embedded

Filter методы
- оценивают признаки без обучения модели, примеры: Pearson, Chi²
- Плюсы: быстрые
- Минусы: не учитывают взаимодействие признаков

Wrapper методы
- обучают модель на разных подмножествах признаков, пример: forward / backward selection
- Минус: очень медленно

Embedded методы
- отбор признаков происходит внутри модели, примеры: Lasso, Ridge, tree models

Как работает Pearson?
- измеряет линейную корреляцию для числовых фич score = |corr(feature, target)|
- чем больше |corr|, тем важнее признак

Как работает Chi‑Square?
- проверяет зависимость feature и target χ² = Σ (Observed − Expected)² / Expected
- используется для категориальных признаков, через подсчет теоретической (ожидаемой) частоты и реальной(наблюдаемой)

Как работает Lasso?
- линейная модель с L1‑регуляризацией

- штраф зануляет неважные признаки

Что такое Permutation Importance?
- оценивает важность через ухудшение метрики (перемешивая числа внутри одной фичи, смотрит поменялась ли от этого метрика)

Алгоритм:
	1.	baseline метрика
	2.	перемешиваем значения признака
	3.	считаем новую метрику

importance = metric_drop

Что такое SHAP?
- метод объяснения моделей на основе Shapley values prediction = baseline + Σ contribution(feature)
- вклад каждого признака показывает его влияние на предсказание
- Смотрим метрику на Baseline части фич. потом со всеми комбинациями интересующих фич и смотрим как метрика от этого меняется.  
  
Когда SHAP реально лучше
	•	когда надо объяснять модель
	•	когда модель не линейная
	•	когда важно понять вклад фичи для конкретного объекта
	•	когда обычный coef_ уже ничего нормально не объясняет

Когда SHAP не лучший выбор
	•	если нужен просто быстрый отбор фич
	•	если данных много и важна скорость
	•	если достаточно грубой оценки важности

# Preprocessing

In [1]:
import numpy as np
import pandas as pd

from sklearn import linear_model
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import r2_score as R2
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit

In [2]:
train = pd.read_json('data/train.json')
test = pd.read_json('data/test.json')

In [3]:
check = set(train.columns) ^ set(test.columns)
check

{'interest_level'}

In [4]:
train = train.drop(columns='interest_level')

In [5]:
data = pd.concat([train, test], axis=0, ignore_index=True)
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 124011 entries, 0 to 124010
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   bathrooms        124011 non-null  float64
 1   bedrooms         124011 non-null  int64  
 2   building_id      124011 non-null  str    
 3   created          124011 non-null  str    
 4   description      124011 non-null  str    
 5   display_address  124011 non-null  str    
 6   features         124011 non-null  object 
 7   latitude         124011 non-null  float64
 8   listing_id       124011 non-null  int64  
 9   longitude        124011 non-null  float64
 10  manager_id       124011 non-null  str    
 11  photos           124011 non-null  object 
 12  price            124011 non-null  int64  
 13  street_address   124011 non-null  str    
dtypes: float64(3), int64(3), object(2), str(6)
memory usage: 13.2+ MB


In [6]:
features = pd.Series([elem.strip() for string in data.features for elem in string ])
names = features.value_counts().head(20)
keys = names.index.tolist()

cols = {f"has_{k}": data.features.map(lambda string, t_k=k: t_k in string).astype("int8") for k in keys}
new_features = pd.DataFrame()
new_features = new_features.assign(**cols)
new_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 124011 entries, 0 to 124010
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   has_Elevator             124011 non-null  int8 
 1   has_Cats Allowed         124011 non-null  int8 
 2   has_Hardwood Floors      124011 non-null  int8 
 3   has_Dogs Allowed         124011 non-null  int8 
 4   has_Doorman              124011 non-null  int8 
 5   has_Dishwasher           124011 non-null  int8 
 6   has_No Fee               124011 non-null  int8 
 7   has_Laundry in Building  124011 non-null  int8 
 8   has_Fitness Center       124011 non-null  int8 
 9   has_Pre-War              124011 non-null  int8 
 10  has_Laundry in Unit      124011 non-null  int8 
 11  has_Roof Deck            124011 non-null  int8 
 12  has_Outdoor Space        124011 non-null  int8 
 13  has_Dining Room          124011 non-null  int8 
 14  has_High Speed Internet  124011 non-null  int8 

In [7]:
df = pd.concat([data.drop(columns='features'), new_features], axis=1)
df = df.select_dtypes(include="number")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 124011 entries, 0 to 124010
Data columns (total 26 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   bathrooms                124011 non-null  float64
 1   bedrooms                 124011 non-null  int64  
 2   latitude                 124011 non-null  float64
 3   listing_id               124011 non-null  int64  
 4   longitude                124011 non-null  float64
 5   price                    124011 non-null  int64  
 6   has_Elevator             124011 non-null  int8   
 7   has_Cats Allowed         124011 non-null  int8   
 8   has_Hardwood Floors      124011 non-null  int8   
 9   has_Dogs Allowed         124011 non-null  int8   
 10  has_Doorman              124011 non-null  int8   
 11  has_Dishwasher           124011 non-null  int8   
 12  has_No Fee               124011 non-null  int8   
 13  has_Laundry in Building  124011 non-null  int8   
 14  has_Fitness Cen

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_log_comparison(
    x: pd.Series,
    y: pd.Series,
    x_name: str | None = None,
    y_name: str | None = None,
    use_log1p: bool = False,
) -> None:
    """
    Строит 4 графика:
    1) x vs y
    2) x vs log(y)
    3) log(x) vs y
    4) log(x) vs log(y)

    Parameters
    ----------
    x : pd.Series
        Фича.
    y : pd.Series
        Таргет.
    x_name : str | None
        Имя оси X. Если None, берётся x.name.
    y_name : str | None
        Имя оси Y. Если None, берётся y.name.
    use_log1p : bool
        Если True, используется np.log1p().
        Если False, используется np.log(), и тогда нужны строго положительные значения.
    """
    df = pd.concat([x, y], axis=1).copy()
    df.columns = ["x", "y"]
    df = df.dropna()

    x_name = x_name or (x.name if x.name is not None else "x")
    y_name = y_name or (y.name if y.name is not None else "y")

    if use_log1p:
        x_log = np.log1p(df["x"])
        y_log = np.log1p(df["y"])
        x_log_label = f"log1p({x_name})"
        y_log_label = f"log1p({y_name})"
    else:
        valid_mask = (df["x"] > 0) & (df["y"] > 0)
        df = df.loc[valid_mask].copy()

        if df.empty:
            raise ValueError("После фильтрации не осталось строк: для np.log() нужны x > 0 и y > 0.")

        x_log = np.log(df["x"])
        y_log = np.log(df["y"])
        x_log_label = f"ln({x_name})"
        y_log_label = f"ln({y_name})"

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].scatter(df["x"], df["y"], alpha=0.6)
    axes[0, 0].set_title(f"{x_name} vs {y_name}")
    axes[0, 0].set_xlabel(x_name)
    axes[0, 0].set_ylabel(y_name)
    axes[0, 0].grid(True)

    axes[0, 1].scatter(df["x"], y_log, alpha=0.6)
    axes[0, 1].set_title(f"{x_name} vs {y_log_label}")
    axes[0, 1].set_xlabel(x_name)
    axes[0, 1].set_ylabel(y_log_label)
    axes[0, 1].grid(True)

    axes[1, 0].scatter(x_log, df["y"], alpha=0.6)
    axes[1, 0].set_title(f"{x_log_label} vs {y_name}")
    axes[1, 0].set_xlabel(x_log_label)
    axes[1, 0].set_ylabel(y_name)
    axes[1, 0].grid(True)

    axes[1, 1].scatter(x_log, y_log, alpha=0.6)
    axes[1, 1].set_title(f"{x_log_label} vs {y_log_label}")
    axes[1, 1].set_xlabel(x_log_label)
    axes[1, 1].set_ylabel(y_log_label)
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_log_comparison(df["bathrooms"], df["price"])

# Train_Test_Split

## Реализация

In [8]:
class my_train_test_split:

    def __init__ (self):
        self.rng = np.random.default_rng(42)

    def split(self, data, test_size = 0.25, shuffle = True, stratify = False):

        X = data.copy()
        n = len(X)
        idx = np.arange(n)
        if shuffle:
            self.rng.shuffle(idx)

        s = n - int(np.ceil(test_size*n))

        train_idx = idx[:s]
        test_idx = idx[s:]
        return X.iloc[train_idx], X.iloc[test_idx]
    

    def split_val(self, data, val_size = 0.15, test_size = 0.25, shuffle = True):

        X = data.copy()
        n = len(X)
        idx = np.arange(n)
        if shuffle:
            self.rng.shuffle(idx)

        s1 = n - int(np.ceil(test_size*n) + np.ceil(val_size*n))
        s2 = n - int(np.ceil(test_size*n))

        train_idx = idx[:s1]
        val_idx = idx[s1:s2]
        test_idx = idx[s2:]
        return X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx]
    
    
    def split_by_date(self, data, date_split, date_col):
        X = data.copy()

        date_split = pd.to_datetime(date_split).date()
        X[date_col] = pd.to_datetime(X[date_col]).dt.date

        
        # X = X.sort_values(date_col, kind='mergesort').reset_index(drop = True)

        left = X.loc[X[date_col] <= date_split]
        right = X.loc[X[date_col] > date_split]
        return left, right


    def split_by_three_date(self, data, validation_date, test_date, date_col):
        X = data.copy()

        validation_date = pd.to_datetime(validation_date).date()
        test_date = pd.to_datetime(test_date).date()
        X[date_col] = pd.to_datetime(X[date_col]).dt.date

        # X = X.sort_values(date_col, kind='mergesort').reset_index(drop = True)

        train = X.loc[X[date_col] <= validation_date]
        val = X.loc[(X[date_col] > validation_date) & (X[date_col] <= test_date)]
        test = X.loc[X[date_col] > test_date]

        return train, val, test
    

my_split = my_train_test_split()

## Train-Test Split

In [9]:
my_train1, my_test1 = my_split.split(df)

print(f"train:{len(my_train1)}, test: {len(my_test1)}\nsum: {len(my_train1) + len(my_test1)}")

train:93008, test: 31003
sum: 124011


In [10]:
my_train2, my_val2, my_test2 = my_split.split_val(df)

print(f"train:{len(my_test2)}, val: {len(my_val2)}, test: {len(my_test2)}\nsum: {len(my_train2) + len(my_test2)+len(my_val2)}")

train:31003, val: 18602, test: 31003
sum: 124011


добавляем колонку даты) 

In [11]:
df['created'] = pd.to_datetime(data.created).dt.date
df.created.info()

<class 'pandas.Series'>
RangeIndex: 124011 entries, 0 to 124010
Series name: created
Non-Null Count   Dtype 
--------------   ----- 
124011 non-null  object
dtypes: object(1)
memory usage: 969.0+ KB


In [12]:
# df.created.tolist()

In [13]:
left, right = my_split.split_by_date(df, '2016/6/15', 'created')

print(f"left:{len(left)}, right: {len(right)}\nsum: {len(left) + len(right)}")

left:104378, right: 19633
sum: 124011


In [14]:
train3, val3, test3 = my_split.split_by_three_date(df, '2016/6/15', '2016/6/23', 'created')

print(f"train:{len(train3)}, val: {len(val3)}, test: {len(test3)}\nsum: {len(train3) + len(val3) + len(test3)}")


train:104378, val: 11494, test: 8139
sum: 124011


# K-Fold

In [15]:
class my_KFold: 

    def __init__(self, k, shuffle = True, random_state = 42):
        self.rng = np.random.default_rng(random_state)
        self.k = k
    
    def split (self, len_x):
        l = len_x
        idx = np.arange(l)
        k = self.k

        q = l//k
        r = l%k

        start_elem = 0
        for i in range(k):
            fold_size = q+1 if i < r else q

            last_elem = start_elem + fold_size
            test_idx = idx[start_elem : last_elem]
            train_idx = np.concatenate([idx[:start_elem], idx[last_elem:]])
            yield train_idx, test_idx
            start_elem = last_elem


    def Group_Split(self, group_field : pd.Series):
        k = self.k
        groups = group_field
        uniq = groups.unique()
        sizes = groups.value_counts()

        folds = [[] for _ in range(k)]
        load = np.zeros(k)

        for g in uniq:
            i = load.argmin()
            folds[i].extend(groups.index[groups == g])
            load[i] += sizes[g]

        for i in range(k):
            train  = []
            [train.extend(folds[j]) for j in range(k) if j!=i]
            yield np.array(train), np.array(folds[i])


    def Strat_split(self, strat_field : pd.Series):
        k = self.k
        col = strat_field 
        uniq = col.unique()
        folds = [[] for _ in range(k)]

        for g in uniq:
            group = col.loc[col == g]
            for i, idx in enumerate(group.index):
                folds[i % k].append(idx)
        
        for i in range(k):
            train  = []
            [train.extend(folds[j]) for j in range(k) if j!=i]
            yield np.array(train), np.array(folds[i])


    def Date_split(self, Date_col : pd.Series):
        k = self.k
        col = Date_col.sort_values(ascending=True)
        l = len(col)
        idx = col.index

        q = l//(k+1)
        r = l%(k+1)

        for i in range(k):
            train_size = r +  q*(i+1)
            train = idx[:train_size]
            test = idx[train_size:train_size+q]
            yield np.array(train), np.array(test)


my_kf = my_KFold(5)

In [16]:
list(my_kf.split(len(df)))

[(array([ 24803,  24804,  24805, ..., 124008, 124009, 124010],
        shape=(99208,)),
  array([    0,     1,     2, ..., 24800, 24801, 24802], shape=(24803,))),
 (array([     0,      1,      2, ..., 124008, 124009, 124010],
        shape=(99209,)),
  array([24803, 24804, 24805, ..., 49602, 49603, 49604], shape=(24802,))),
 (array([     0,      1,      2, ..., 124008, 124009, 124010],
        shape=(99209,)),
  array([49605, 49606, 49607, ..., 74404, 74405, 74406], shape=(24802,))),
 (array([     0,      1,      2, ..., 124008, 124009, 124010],
        shape=(99209,)),
  array([74407, 74408, 74409, ..., 99206, 99207, 99208], shape=(24802,))),
 (array([    0,     1,     2, ..., 99206, 99207, 99208], shape=(99209,)),
  array([ 99209,  99210,  99211, ..., 124008, 124009, 124010],
        shape=(24802,)))]

In [17]:
list(my_kf.Group_Split(df.created))

[(array([     1,     11,     29, ..., 123920, 123975, 123995],
        shape=(99886,)),
  array([     0,     17,     18, ..., 110179, 116084, 122219],
        shape=(24125,))),
 (array([     0,     17,     18, ..., 123920, 123975, 123995],
        shape=(99001,)),
  array([     1,     11,     29, ..., 123922, 123947, 123994],
        shape=(25010,))),
 (array([     0,     17,     18, ..., 123920, 123975, 123995],
        shape=(99839,)),
  array([     2,     22,     33, ..., 123878, 123951, 124010],
        shape=(24172,))),
 (array([     0,     17,     18, ..., 123920, 123975, 123995],
        shape=(98562,)),
  array([     3,     20,     38, ..., 123948, 123959, 123983],
        shape=(25449,))),
 (array([     0,     17,     18, ..., 123948, 123959, 123983],
        shape=(98756,)),
  array([     4,      5,      9, ..., 123920, 123975, 123995],
        shape=(25255,)))]

In [18]:
list(my_kf.Strat_split(df.created))

[(array([    17,     57,    105, ..., 123788, 123920, 106076],
        shape=(99175,)),
  array([     0,     49,     97, ..., 123975,  44454, 110179],
        shape=(24836,))),
 (array([     0,     49,     97, ..., 123788, 123920, 106076],
        shape=(99192,)),
  array([    17,     57,    105, ..., 123995,  45569, 116084],
        shape=(24819,))),
 (array([     0,     49,     97, ..., 123788, 123920, 106076],
        shape=(99206,)),
  array([    18,     58,    113, ..., 123893,  46731, 122219],
        shape=(24805,))),
 (array([     0,     49,     97, ..., 123788, 123920, 106076],
        shape=(99227,)),
  array([    31,     84,    121, ..., 123778, 123918,  46931],
        shape=(24784,))),
 (array([     0,     49,     97, ..., 123778, 123918,  46931],
        shape=(99244,)),
  array([    47,     92,    147, ..., 123788, 123920, 106076],
        shape=(24767,)))]

In [19]:
list(my_kf.Date_split(df.created))

[(array([ 44454, 110179, 122219, ...,  46625,  34767, 120814],
        shape=(20671,)),
  array([ 99212, 119896,  34590, ...,  95469,  26749,  29567],
        shape=(20668,))),
 (array([ 44454, 110179, 122219, ...,  95469,  26749,  29567],
        shape=(41339,)),
  array([95123, 20779, 21402, ..., 90895, 95678, 97658], shape=(20668,))),
 (array([ 44454, 110179, 122219, ...,  90895,  95678,  97658],
        shape=(62007,)),
  array([97039, 18440, 97055, ..., 55988,    11, 69486], shape=(20668,))),
 (array([ 44454, 110179, 122219, ...,  55988,     11,  69486],
        shape=(82675,)),
  array([ 4253, 70052, 55296, ..., 51738, 51725,   954], shape=(20668,))),
 (array([ 44454, 110179, 122219, ...,  51738,  51725,    954],
        shape=(103343,)),
  array([16736, 59668, 59244, ..., 61579, 14867,  6511], shape=(20668,)))]

# Cross-Validation comparison

In [20]:
my_cv = my_KFold(5)

In [21]:
df['price'] = data.price

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 124011 entries, 0 to 124010
Data columns (total 27 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   bathrooms                124011 non-null  float64
 1   bedrooms                 124011 non-null  int64  
 2   latitude                 124011 non-null  float64
 3   listing_id               124011 non-null  int64  
 4   longitude                124011 non-null  float64
 5   price                    124011 non-null  int64  
 6   has_Elevator             124011 non-null  int8   
 7   has_Cats Allowed         124011 non-null  int8   
 8   has_Hardwood Floors      124011 non-null  int8   
 9   has_Dogs Allowed         124011 non-null  int8   
 10  has_Doorman              124011 non-null  int8   
 11  has_Dishwasher           124011 non-null  int8   
 12  has_No Fee               124011 non-null  int8   
 13  has_Laundry in Building  124011 non-null  int8   
 14  has_Fitness Cen

In [23]:
def cv_trainer(df, generator):

    test_preds, train_preds = [],[]
    X = df.drop(columns='price')
    X['created'] = pd.to_datetime(X['created'])
    X['created'] = X['created'].astype('int64')
    y = df['price']

    for train_idx, test_idx in generator:
        x_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        x_test = X.iloc[test_idx]
        y_test = y.iloc[test_idx]

        model = linear_model.LinearRegression()
        model.fit(x_train, y_train)

        p_train = model.predict(x_train)
        p_test = model.predict(x_test)

        train_mae, test_mae = MAE(y_train, p_train), MAE(y_test, p_test)
        print(f"MAE - train: {train_mae}, test: {test_mae}")

        train_preds.append(train_mae)
        test_preds.append(test_mae)
    
    return sum(train_preds) / len(train_preds), sum(test_preds) / len(test_preds)

### Classic KFold

In [24]:
cv_trainer(df, my_cv.split(len(df)))

MAE - train: 960.4539515192972, test: 1236.607096529672
MAE - train: 1123.8462615628252, test: 945.8512446628243
MAE - train: 990.1184632259921, test: 1224.136072528245
MAE - train: 1125.615583206306, test: 957.6611556183253
MAE - train: 1121.531148196599, test: 951.7244165657086


(1064.313081542204, 1063.1959971809551)

In [25]:
kf = KFold(n_splits=5, shuffle=False)
cv_trainer(df, kf.split(df))

MAE - train: 960.4539515192972, test: 1236.607096529672
MAE - train: 1123.8462615628252, test: 945.8512446628243
MAE - train: 990.1184632259921, test: 1224.136072528245
MAE - train: 1125.615583206306, test: 957.6611556183253
MAE - train: 1121.531148196599, test: 951.7244165657086


(1064.313081542204, 1063.1959971809551)

### Grouped KFold

In [26]:
cv_trainer(df, my_kf.Group_Split(df.created))

MAE - train: 1122.352969418124, test: 1000.2770948030021
MAE - train: 1077.3307042615477, test: 976.0912254831164
MAE - train: 1100.3263186797778, test: 1024.7648350588543
MAE - train: 925.0215051683278, test: 1292.843602152131
MAE - train: 1103.7551415178373, test: 1037.6158431136266


(1065.757327809123, 1066.318520122146)

In [27]:
g_kf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
cv_trainer(df, g_kf.split(df, groups=df.created))

MAE - train: 1122.502129834085, test: 978.4641120946394
MAE - train: 1110.606768782785, test: 971.0514512175529
MAE - train: 943.5436806899473, test: 1280.6454270631002
MAE - train: 1060.3563965280102, test: 1126.461942733582
MAE - train: 1093.011119128412, test: 1012.951154284089


(1066.004018992648, 1073.9148174785928)

### Strat KFold

In [28]:
cv_trainer(df, my_kf.Strat_split(df.price))

MAE - train: 886.5443850193536, test: 1384.3215023736102
MAE - train: 1102.855920579308, test: 1012.4615078831316
MAE - train: 1108.9458312949894, test: 992.1048867067007
MAE - train: 1111.326754538585, test: 981.7034584106074
MAE - train: 1110.0467765491637, test: 929.5583141604293


(1063.94393359628, 1060.0299339068958)

In [29]:
st_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_trainer(df, st_kf.split(df, df.price))

/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


MAE - train: 1057.5413124025724, test: 1093.2932943154065
MAE - train: 1073.815022372695, test: 1057.1591052683118
MAE - train: 1103.2796923810004, test: 1012.9051680129232
MAE - train: 1101.057192332635, test: 1022.7190206877718
MAE - train: 980.690139011769, test: 1132.8486750276932


(1063.2766717001343, 1063.7850526624213)

### Time KFold

In [30]:
cv_trainer(df, my_kf.Date_split(df.created))

MAE - train: 871.9342797247008, test: 867.2406025581871
MAE - train: 869.0179465726306, test: 895.1103713982341
MAE - train: 878.476846374349, test: 923.7921810560849
MAE - train: 897.4327668543569, test: 966.0756249023151
MAE - train: 918.2371102248525, test: 1410.1654209075161


(887.0197899501779, 1012.4768401644675)

In [31]:
t_kf = TimeSeriesSplit(n_splits=5)
time_df = df.sort_values("created")
cv_trainer(df, t_kf.split(time_df))

MAE - train: 1481.5156137365223, test: 1348.4668451170994
MAE - train: 1227.8825951843837, test: 1222.8558783242092
MAE - train: 1231.0578229199646, test: 1211.8763635805547
MAE - train: 1185.0127242678884, test: 994.0609732035769
MAE - train: 1111.0802508981806, test: 939.9882824125166


(1247.3098014013879, 1143.4496685275913)

# Feature Selection

### Lasso with normalize, split 60/20/20

In [32]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler


In [33]:
train, val, test = my_split.split_val(df, val_size=0.2, test_size=0.2)

l, r = train['price'].quantile([0.01, 0.99])
train = train.loc[(train.price>=l)&(train.price<=r)]
val = val.loc[(val.price>=l)&(val.price<=r)]
# test = test.loc[(test.price>=l)&(test.price<=r)]

l, r = train['bathrooms'].quantile([0.01, 0.99])
train = train.loc[(train.bathrooms>=l)&(train.bathrooms<=r)]
val = val.loc[(val.bathrooms>=l)&(val.bathrooms<=r)]
# test = test.loc[(test.bathrooms>=l)&(test.bathrooms<=r)]
test = test.loc[test['price'] != test["price"].max()]
test = test.loc[test['price'] != test["price"].max()]
test = test.loc[test['price'] != test["price"].max()]
X_train = train.drop(columns='price')
X_train['created'] = pd.to_datetime(train['created'])
X_train['created'] = X_train['created'].astype('int64')
y_train = train['price']

X_val = val.drop(columns='price')
X_val['created'] = pd.to_datetime(val['created'])
X_val['created'] = X_val['created'].astype('int64')
y_val = val['price']

X_test = test.drop(columns='price')
X_test['created'] = pd.to_datetime(test['created'])
X_test['created'] = X_test['created'].astype('int64')
y_test = test['price']


scaler = StandardScaler()

# X_train_scaled = scaler.fit_transform(X_train)
# X_val_scaled = scaler.transform(X_val)
# X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

lasso = linear_model.Lasso(alpha=0.001)
lasso.fit(X_train, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [34]:
y_test.max()

np.int64(85000)

lasso

In [35]:
print(f"MAE_train: {MAE(y_train, lasso.predict(X_train))}")
print(f"R2_train: {R2(y_train, lasso.predict(X_train))}")
print(f"MAE_val: {MAE(y_val, lasso.predict(X_val))}")
print(f"R2_val: {R2(y_val, lasso.predict(X_val))}")
print(f"MAE_test: {MAE(y_test, lasso.predict(X_test))}")
print(f"R2_test: {R2(y_test, lasso.predict(X_test))}")

MAE_train: 699.4861335280291
R2_train: 0.5789959546665399
MAE_val: 690.6474202372196
R2_val: 0.5860158575755783
MAE_test: 815.1774278192679
R2_test: 0.48452734764761407


In [36]:
top = pd.Series(lasso.coef_, index=X_train.columns).abs().sort_values(ascending=False)
top_ten = top.head(10)
top_ten

bathrooms                  720.338660
bedrooms                   476.451767
has_Doorman                291.732969
longitude                  214.730422
latitude                   209.630107
has_Laundry in Unit        167.455870
has_Elevator               114.739330
has_Fitness Center         101.347484
has_No Fee                  87.525755
has_Laundry in Building     86.602924
dtype: float64

In [37]:
lasso_10 = linear_model.Lasso(alpha=10)
X_train_10 = X_train[top_ten.index]
lasso_10.fit(X_train_10, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",10
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


top10 after lasso drop

In [38]:
print(f"train: {MAE(y_train, lasso_10.predict(X_train_10))}")
print(f"val: {MAE(y_val, lasso_10.predict(X_val[top_ten.index]))}")
print(f"MAE_train: {MAE(y_test, lasso_10.predict(X_test[top_ten.index]))}")
print(f"R2_train: {R2(y_test, lasso_10.predict(X_test[top_ten.index]))}")

train: 710.3660837076152
val: 700.276758709895
MAE_train: 825.5975842184945
R2_train: 0.4783704030674235


In [39]:
class Feature_Selection: 

    def __init__(self):
        self.rng = np.random.default_rng(42)


    def select_top_features(self, X, y, top_n=10):
        nan_score = 1 - X.isna().mean()
        corr_score = X.corrwith(y).abs()
        return (nan_score * corr_score).nlargest(top_n).index


    def select_top_permutation_features(self, model, X, y, top_n=10):
        X = X.copy()
        base_score = ((y - model.predict(X)) ** 2).mean()
        scores = dict()

        for col in X.columns:
            X_perm = X.copy()
            X_perm[col] = self.rng.permutation(X_perm[col].values)
            perm_score = ((y - model.predict(X_perm)) ** 2).mean()
            scores[col] = perm_score - base_score

        return pd.Series(scores).nlargest(top_n).index

selector = Feature_Selection()

Apply non+corr method to feature set and take top 10 features, refit model and measure quality.

In [40]:
top_10 = selector.select_top_features(X_train, y_train)
print(top_10)

lasso.fit(X_train[top_10], y_train)
pred = lasso.predict(X_test[top_10])


print("R2:", R2(y_test, pred))
print("MAE:", MAE(y_test, pred))

Index(['bathrooms', 'bedrooms', 'has_Doorman', 'has_Laundry in Unit',
       'has_Fitness Center', 'has_Dishwasher', 'has_Elevator',
       'has_Dining Room', 'has_Outdoor Space', 'has_Laundry in Building'],
      dtype='str')
R2: 0.4784266934405643
MAE: 826.151787585121


permutation top

In [41]:
lasso.fit(X_train, y_train)

top_10 = selector.select_top_permutation_features(lasso, X_train, y_train, top_n=10)
print(top_10)
lasso.fit(X_train[top_10], y_train)
pred = lasso.predict(X_test[top_10])
print("R2:", R2(y_test, pred))
print("MAE:", MAE(y_test, pred))

Index(['bathrooms', 'bedrooms', 'has_Doorman', 'longitude', 'latitude',
       'has_Laundry in Unit', 'has_Elevator', 'has_Fitness Center',
       'has_No Fee', 'has_Laundry in Building'],
      dtype='str')
R2: 0.4809728609583075
MAE: 823.3255143275761


In [42]:
import shap

lasso.fit(X_train, y_train)

explainer = shap.Explainer(lasso, X_train)
shap_values = explainer(X_train)
shap_values.values.mean(axis=0)

/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


array([ 28.08575625,  50.20535659,   3.99575905,  -0.7042773 ,
        -4.31848218,   5.20210437,   1.11865451,   2.88778179,
        -4.63525293, -23.30202292,  -0.70467085,   4.20893299,
         6.97451869, -13.78188638,   0.49117456, -14.98900711,
         1.78706707,  -1.382851  ,  -2.10378439,   0.99111398,
         0.30593085,   0.18258243,  -8.14248689,   1.81403627,
         2.0940314 ,  -0.514281  ])

In [43]:
top_10 = (pd.Series(abs(shap_values.values).mean(axis=0), index=X_train.columns).nlargest(10).index)
print(top_10)

lasso.fit(X_train[top_10], y_train)
pred = lasso.predict(X_test[top_10])

print("R2:", R2(y_test, pred))
print("MAE:", MAE(y_test, pred))

Index(['bathrooms', 'bedrooms', 'has_Doorman', 'has_Laundry in Unit',
       'has_Elevator', 'has_Fitness Center', 'has_No Fee',
       'has_Laundry in Building', 'has_Dishwasher', 'has_Hardwood Floors'],
      dtype='str')
R2: 0.48141067811746974
MAE: 822.2482137193892


In [44]:
el = linear_model.ElasticNet()
el.fit(X_train[top_10], y_train)
pred = el.predict(X_test[top_10])

print("R2:", R2(y_test, pred))
print("MAE:", MAE(y_test, pred))

R2: 0.41512946706774523
MAE: 862.0571787243838


#### Compare all validation schemes. Choose the best one. Explain your choice.
ну всё буквально в порядке возрастания:
time: simple < perm < shap
metrics: simple < perm < shap

сам выбрал бы наверное perm на больших данных, тут норм и с shap

# Hyperparameter optimization

In [45]:
from itertools import product


class Hyperparam_Search: 

    def __init__(self, random_state = 42):
        self.rng = np.random.default_rng(random_state)


    def grid_search(self, X_train, y_train, X_val, y_val, 
                    model_class = linear_model.ElasticNet, 
                    score_func = MAE, 
                    **params
                    ):
        keys = params.keys()
        values = params.values()

        best_score = None
        best_params = None

        for combined in product(*values):
            current_params = dict(zip(keys, combined))
            model = model_class(**current_params)
            model.fit(X_train, y_train)
            pred = model.predict(X_val)
            score = score_func(y_val, pred)

            if best_score is None or score < best_score:
                best_score = score
                best_params = current_params

        return best_params, best_score


    def random_search(self, X_train, y_train, X_val, y_val,
        model_class=linear_model.ElasticNet,
        score_func=MAE,
        n_iter=50,
        **params
    ):
        keys = list(params.keys())
        values = list(params.values())

        best_score = None
        best_params = None

        for _ in range(n_iter):
            current_params = { key: self.rng.choice(value_list) for key, value_list in zip(keys, values) }

            model = model_class(**current_params)
            model.fit(X_train, y_train)
            pred = model.predict(X_val)
            score = score_func(y_val, pred)

            if best_score is None or score < best_score:
                best_score = score
                best_params = current_params

        return best_params, best_score

    # def grid_search_cv(self, df, generator, **params):

    #     X = df.drop(columns='price')
    #     X['created'] = pd.to_datetime(X['created'])
    #     X['created'] = X['created'].astype('int64')
    #     y = df['price']

    #     for train_idx, val_idx in generator:
    #         X_train = X.iloc[train_idx]
    #         y_train = y.iloc[train_idx]
    #         X_val = X.iloc[val_idx]
    #         y_val = y.iloc[val_idx]
    #         yield self.grid_search(self, X_train, y_train, X_val, y_val, 
    #                 model_class = linear_model.ElasticNet, 
    #                 score_func = MAE, 
    #                 **params
    #                 )



searcher = Hyperparam_Search()

In [46]:
# Enet = linear_model.ElasticNet()
pepe = searcher.grid_search(X_train, y_train, X_val, y_val,
                    alpha=[0.001, 0.01, 0.1, 1, 10, 100],
                    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                    max_iter=[1000, 3000, 5000]
                    )
pepe

({'alpha': 0.001, 'l1_ratio': 0.9, 'max_iter': 1000}, 690.6507499717793)

In [47]:
pepe = searcher.grid_search(X_train, y_train, X_val, y_val, 
                    alpha=[0.001, 0.0, 0.0011, 0.0012, 0.0009, 0.0008],
                    l1_ratio=[0.05, 0.1, 0.09, 0.11, 0.08, 0.12],
                    max_iter=[500, 750, 1000, 1250, 1500],
                    tol=[0.0001, 0.001],
                    )
pepe

/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/base.py:1336: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.726e+10, tolerance: 1.770e+07
Linear regression m

({'alpha': 0.0, 'l1_ratio': 0.05, 'max_iter': 750, 'tol': 0.0001},
 690.6473178598775)

In [48]:
pepe = searcher.random_search(X_train, y_train, X_val, y_val, 
                    alpha=[0.001, 0.0, 0.0011, 0.0012, 0.0009, 0.0008],
                    l1_ratio=[0.05, 0.1, 0.09, 0.11, 0.08, 0.12],
                    max_iter=[500, 750, 1000, 1250, 1500],
                    random_state = 42
                    )
pepe

/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/base.py:1336: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.726e+10, tolerance: 1.770e+07
Linear regression m

({'alpha': np.float64(0.0),
  'l1_ratio': np.float64(0.05),
  'max_iter': np.int64(1000),
  'random_state': 40},
 690.6473178598775)

In [49]:
Enet = linear_model.ElasticNet(alpha=0.0, max_iter=750, random_state = 42)
Enet.fit(X_train, y_train)
pred = Enet.predict(X_test)

print("R2:", R2(y_test, pred))
print("MAE:", MAE(y_test, pred))

/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/base.py:1336: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(


R2: 0.48452766379166057
MAE: 815.1771558588267


/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML3_Validation/ml3.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.726e+10, tolerance: 1.770e+07
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


In [50]:
Enet = linear_model.LinearRegression()
Enet.fit(X_train, y_train)
pred = Enet.predict(X_test)

print("R2:", R2(y_test, pred))
print("MAE:", MAE(y_test, pred))

R2: 0.4845276637916749
MAE: 815.1771558588248


# Optuna

функция которая у меня кросс-валидацию учит (чутка доработанная)

In [51]:
def cv_score(df, generator, model_class, score_func, **params):
    scores = []

    X = df.drop(columns="price").copy()
    X["created"] = pd.to_datetime(X["created"])
    X["created"] = X["created"].astype("int64")
    y = df["price"]

    for train_idx, test_idx in generator:
        x_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        x_test = X.iloc[test_idx]
        y_test = y.iloc[test_idx]

        model = model_class(**params)
        model.fit(x_train, y_train)

        pred = model.predict(x_test)
        score = score_func(y_test, pred)
        scores.append(score)

    return sum(scores) / len(scores)

In [52]:
import optuna

def tunez(trial):
    alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 1e-2, 1.0)

    score = cv_score(
        df=df,
        generator=my_kf.Strat_split(df.price),
        model_class=linear_model.ElasticNet,
        score_func=MAE,
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=2000,
        random_state=42,
    )

    return score

In [53]:
study = optuna.create_study(direction="minimize")
study.optimize(tunez, n_trials=30)
study.best_params

[I 2026-03-14 10:32:04,332] A new study created in memory with name: no-name-31883744-5cf3-4927-836b-6cdf487d3bab
[I 2026-03-14 10:32:05,258] Trial 0 finished with value: 1138.2475762580743 and parameters: {'alpha': 7.9019274092422656, 'l1_ratio': 0.8761021505799068}. Best is trial 0 with value: 1138.2475762580743.
[I 2026-03-14 10:32:06,559] Trial 1 finished with value: 1059.9840366126707 and parameters: {'alpha': 3.6530667893587005e-05, 'l1_ratio': 0.21014175358169657}. Best is trial 1 with value: 1059.9840366126707.
[I 2026-03-14 10:32:07,989] Trial 2 finished with value: 1059.9984583823634 and parameters: {'alpha': 2.4220228929224195e-05, 'l1_ratio': 0.1832103855594428}. Best is trial 1 with value: 1059.9840366126707.
[I 2026-03-14 10:32:09,361] Trial 3 finished with value: 1060.0185500015032 and parameters: {'alpha': 1.1551131750228174e-05, 'l1_ratio': 0.3815202960877561}. Best is trial 1 with value: 1059.9840366126707.
[I 2026-03-14 10:32:10,432] Trial 4 finished with value: 1048

{'alpha': 0.0910529632746906, 'l1_ratio': 0.11097464477507406}

In [54]:
tuna_model = linear_model.ElasticNet(
    **study.best_params,
    max_iter=10_000,
    random_state=42,
)
tuna_model.fit(X_train, y_train)
pred = tuna_model.predict(X_test)
print("R2:", R2(y_test, pred))
print("MAE:", MAE(y_test, pred))

R2: 0.4735183180580913
MAE: 818.8896955847493


Grid_search давал  
R2: 0.5871037883335661  
MAE: 696.0429108522518